# Graph Programming — 02: Depth-First Search (DFS)

**DFS** explores as far as possible along a branch before backtracking.  
Think of it as: *go deep, then come back.*

```
Graph:       DFS from 0 visits: 0 → 1 → 3 → 4 → 2
  0
 / \
1   2
|   
3
|
4
```

## When to use DFS
- Find **all paths** between two nodes
- Detect **cycles**
- Find **connected components**
- **Topological sort**
- Islands problems
- Backtracking (permutations, subsets)

## Two implementations
1. **Recursive** — cleaner, uses the call stack
2. **Iterative** — explicit stack, avoids recursion limit

---

In [1]:
from collections import defaultdict

# Build sample graph
#   0 -- 1 -- 3
#   |    |    |
#   2    4 ---+

graph = defaultdict(list, {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0],
    3: [1, 4],
    4: [1, 3]
})

In [2]:
# ============================================================
# RECURSIVE DFS
# ============================================================
# Template (memorize):
#
#   def dfs(node, visited):
#       if node in visited: return
#       visited.add(node)
#       for neighbor in graph[node]:
#           dfs(neighbor, visited)

def dfs_recursive(graph, start):
    visited = set()
    order = []

    def dfs(node):
        if node in visited:
            return
        visited.add(node)
        order.append(node)
        for neighbor in graph[node]:
            dfs(neighbor)

    dfs(start)
    return order

print("Recursive DFS from 0:", dfs_recursive(graph, 0))

Recursive DFS from 0: [0, 1, 3, 4, 2]


In [ ]:
# ============================================================
# ITERATIVE DFS (explicit stack)
# ============================================================
# Template:
#
#   stack = [start]
#   visited = set()
#   while stack:
#       node = stack.pop()        # LIFO = depth-first
#       if node in visited: continue
#       visited.add(node)
#       for neighbor in graph[node]:
#           stack.append(neighbor)

def dfs_iterative(graph, start):
    visited = set()
    order = []
    stack = [start]

    while stack:
        node = stack.pop()        # pop from end = LIFO
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                stack.append(neighbor)

    return order

print("Iterative DFS from 0:", dfs_iterative(graph, 0))

---
## Problem 1: Count Connected Components

Given `n` nodes and a list of edges, find how many **separate connected components** exist.

```
Example:
  n=6, edges=[[0,1],[1,2],[3,4]]
  Components: {0,1,2}, {3,4}, {5}
  Answer: 3
```

**Strategy:** Iterate all nodes. If unvisited, run DFS and count it as one component.

In [ ]:
def count_components(n, edges):
    # Build adjacency list
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set()
    count = 0

    def dfs(node):
        if node in visited:
            return
        visited.add(node)
        for neighbor in graph[node]:
            dfs(neighbor)

    for node in range(n):          # try every node as a start
        if node not in visited:    # only unvisited = new component
            dfs(node)
            count += 1

    return count

# Test
print(count_components(6, [[0,1],[1,2],[3,4]]))  # 3
print(count_components(5, [[0,1],[1,2],[2,3],[3,4]]))  # 1 (chain)

---
## Problem 2: Has Path (DFS path finding)

Given a **directed** graph, does a path exist from `src` to `dst`?

In [ ]:
def has_path(graph, src, dst, visited=None):
    if visited is None:
        visited = set()
    if src == dst:
        return True
    if src in visited:
        return False
    visited.add(src)
    for neighbor in graph[src]:
        if has_path(graph, neighbor, dst, visited):
            return True
    return False

directed = defaultdict(list, {
    0: [1, 2],
    1: [3],
    2: [4],
    3: [],
    4: []
})

print(has_path(directed, 0, 3))  # True
print(has_path(directed, 0, 4))  # True
print(has_path(directed, 1, 4))  # False

---
## Problem 3: Cycle Detection in Undirected Graph

A cycle exists if you can revisit a node that isn't your direct parent.

**Key trick:** Track `parent` so you don't immediately go back the way you came.

In [ ]:
def has_cycle_undirected(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set()

    def dfs(node, parent):
        visited.add(node)
        for neighbor in graph[node]:
            if neighbor == parent:       # don't go back where we came from
                continue
            if neighbor in visited:      # visited via different path = cycle!
                return True
            if dfs(neighbor, node):
                return True
        return False

    for node in range(n):
        if node not in visited:
            if dfs(node, -1):
                return True
    return False

print(has_cycle_undirected(4, [[0,1],[1,2],[2,3],[3,1]]))  # True (1-2-3-1)
print(has_cycle_undirected(4, [[0,1],[1,2],[2,3]]))         # False (just a chain)

---
## DFS on a Grid — Number of Islands (preview)

This is covered fully in `04_islands.ipynb`, but here's the core idea:
- Each `'1'` cell is land
- DFS from a land cell "sinks" the entire island (marks all connected land as visited)
- Count how many times you start a new DFS = number of islands

In [ ]:
def num_islands_preview(grid):
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        # Base cases: out of bounds or water — stop
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != '1':
            return
        grid[r][c] = '#'   # mark visited by "sinking" the land
        dfs(r+1, c)
        dfs(r-1, c)
        dfs(r, c+1)
        dfs(r, c-1)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                count += 1
                dfs(r, c)    # sink the whole island
    return count

grid = [
    ['1','1','0','0'],
    ['1','0','0','1'],
    ['0','0','1','1']
]
print(num_islands_preview(grid))  # 3

---
## DFS Complexity

| | Time | Space |
|---|---|---|
| Graph (adj list) | O(V + E) | O(V) for visited + call stack |
| Grid | O(rows × cols) | O(rows × cols) worst case stack |

## Common Mistakes
1. **Forgetting `visited`** — infinite loop on cycles
2. **Modifying graph during DFS** — ok for grids (mark visited), careful otherwise
3. **Python recursion limit** — `sys.setrecursionlimit(10000)` for large inputs, or use iterative

**Next:** `03_bfs.ipynb` — Breadth-First Search